# Revisión de historiales de identidad con dos relojes

**Pregunta.** En una evaluación point-in-time, ¿qué sucede si se filtra la vigencia antes de seleccionar la revisión más nueva disponible? El repo ya gatea snapshots prospectivos; este incremento prueba la regla de versión completa para historiales. Datos diseñados, no muestra empírica.

Madurez: prototipo experimental local. La novedad es de ingeniería acumulativa y no una nueva teoría temporal.

## Semántica y derivación

Para entidad $i$, cada revisión completa $r$ lleva disponibilidad $a(r)$, inicio de vigencia $e(r)$ y fin exclusivo $u(r)$. Para corte de conocimiento $A$ y fecha del hecho $T$, primero elegimos

\[r^*(i,A)=\arg\max_{r:a(r)\le A} a(r),\]

Luego emitimos la revisión solo si $e(r^*)\le T<u(r^*)$, tomando $u=+\infty$ si queda abierto. El orden importa bajo semántica de reemplazo completa: una revisión más reciente puede retirar o postergar la anterior; no debemos reactivar una versión antigua solo porque su intervalo anterior aparentaba abierto. Intervalos UTC semiabiertos; fechas naive y empates recientes son errores.

In [1]:
from src.temporal_revisions import Revision, select_full_replacements, filter_validity_first
from src.revision_experiment import stamp, histories, oracle
from datetime import datetime, timezone, timedelta
from IPython.display import display
import json, csv, gzip


## Contraejemplo mínimo y baseline adversarial

El baseline `validity_first_ablation` aplica vigencia y solo después toma el registro disponible más reciente. Esta ablation es deliberadamente incompatible con full replacement; permite cuantificar qué cambia y debe reemplazarse por la semántica del productor si los eventos fueran deltas.

In [2]:
history=[Revision('x','r0',stamp(0),stamp(0),None,'old'), Revision('x','r1',stamp(3),stamp(5),stamp(9),'new')]
a=select_full_replacements(history,as_of=stamp(4),identity_at=stamp(2))[0]
b=filter_validity_first(history,as_of=stamp(4),identity_at=stamp(2))[0]
print('latest then interval:', a)
print('filter interval then latest (ablation):', b)
assert a.revision_id=='r1' and a.status=='not-effective-at-identity-time'
assert b.revision_id=='r0' and b.status=='eligible'

latest then interval: Selection(identity='x', revision_id='r1', value='new', status='not-effective-at-identity-time', available_at='2026-01-04T00:00:00+00:00', effective_at='2026-01-06T00:00:00+00:00', valid_until='2026-01-10T00:00:00+00:00')
filter interval then latest (ablation): Selection(identity='x', revision_id='r0', value='old', status='eligible', available_at='2026-01-01T00:00:00+00:00', effective_at='2026-01-01T00:00:00+00:00', valid_until=None)


## Matriz de conformidad y política de abstención

Cada fixture sintético compara disponibilidad $A$ y fecha del hecho $T$. La referencia independiente ordena por `available_at` y valida el primer registro, sin invocar el módulo productivo. `abstain_all` es un control seguro pero inútil: su cero falso-admisión viene con falso-rechazo alto. `availability_only` prueba que el reloj transaccional por sí solo no basta.

In [3]:
with open('results/temporal1103/summary.json') as f: summary=json.load(f)
for policy, values in summary.items(): print(policy, values)
assert summary['latest_available_then_validity']['false_admit']==0
assert summary['validity_first_ablation']['false_admit']>0

abstain_all {'admitted': 0, 'false_admit': 0, 'false_reject': 570, 'queries': 1080}
availability_only_ablation {'admitted': 1080, 'false_admit': 510, 'false_reject': 0, 'queries': 1080}
ignore_availability_ablation {'admitted': 360, 'false_admit': 72, 'false_reject': 282, 'queries': 1080}
latest_available_then_validity {'admitted': 570, 'false_admit': 0, 'false_reject': 0, 'queries': 1080}
oracle {'admitted': 570, 'false_admit': 0, 'false_reject': 0, 'queries': 1080}
validity_first_ablation {'admitted': 1080, 'false_admit': 510, 'false_reject': 0, 'queries': 1080}
design {'histories': 36, 'queries': 1080, 'output_rows': 6480, 'synthetic': True, 'seeds': [1103]}


## Sensibilidad temporal (historia diseñada)

La figura es recuento de admisiones válidas de un panel generado de 36 historiales. Diferentes combinaciones de fecha de corte y fecha del hecho cambian la elegibilidad; el punto es revelar la resurrección por orden incorrecto, no estimar prevalencia externa. El CSV completo conserva 1,080 consultas por cada una de seis políticas.

In [4]:
import matplotlib.pyplot as plt
from collections import defaultdict
with gzip.open('results/temporal1103/raw.csv.gz', 'rt', newline='') as f: rows=list(csv.DictReader(f))
policies=['latest_available_then_validity','validity_first_ablation','availability_only_ablation','abstain_all']
labels=['latest→validity','validity→latest','available only','abstain all']
false_admit=[sum(r['policy']==p and r['false_admit']=='True' for r in rows) for p in policies]
false_reject=[sum(r['policy']==p and r['false_reject']=='True' for r in rows) for p in policies]
x=list(range(len(labels))); width=.36
fig,ax=plt.subplots(figsize=(8,4.5),layout='constrained')
ax.bar([i-width/2 for i in x], false_admit, width, label='Falsas admisiones',color='#b75b38')
ax.bar([i+width/2 for i in x], false_reject, width, label='Falsos rechazos',color='#477da8')
ax.set_xticks(x,labels,rotation=10)
ax.set_ylabel('Consultas de fixture (conteo de 1,080)')
ax.set_title('Orden temporal: error de fuga frente a costo de abstención')
ax.legend(frameon=False); ax.grid(axis='y',alpha=.25)
for series,offset in ((false_admit,-width/2),(false_reject,width/2)):
    for i,y in enumerate(series):
        if y: ax.text(i+offset,y+8,str(y),ha='center',fontsize=9)
fig.savefig('results/temporal1103/admissions.png',dpi=160)
display(fig)

<Figure size 800x450 with 1 Axes>

## Resultados, límites y decisión

H1 es exactitud de implementación en la grilla: todas las 1,080 consultas de regla propuesta deben coincidir con oracle independiente y tener cero falsos admitidos. El incremento no demuestra la corrección del contrato de reemplazo para cualquier base de datos; sí produce un contraejemplo reproducible a la ablation. No valida identidad adjudicada, fechas de fuente ni features posteriores; no es un motor bitemporal. No mide predictive performance ni inferencia estadística.

**Continuar** si un upstream acredita historia completa y reloj transaccional fiable; **ajustar** si conserva deltas o permite intervalos solapados; **cerrar claims de seguridad temporal** si solo hay timestamps auto-declarados. Sin esos insumos, el componente queda en prototipo sintético ejecutado.